### Prepare the Environment

In [ ]:
!pip install -q python-dotenv

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

In [ ]:
load_dotenv()

# Load environment variables
password = os.getenv("RDI_PASSWORD")
SOURCE_DB_IP = os.getenv("SOURCE_DB_IP")
SOURCE_DB_PASSWORD = os.getenv("SOURCE_DB_PASSWORD")
TARGET_DB_URL = os.getenv("TARGET_DB_URL")
TARGET_DB_PORT = os.getenv("TARGET_DB_PORT")
TARGET_DB_PASSWORD = os.getenv("TARGET_DB_PASSWORD")

In [ ]:
base_url = "http://api.rdi.redis-demo.com"
port = 80
access_token = "NOT_AUTHENTICATED"

### Helper Functions

In [ ]:
# GET action
def get_action(base_url, action_id, access_token):

    url = f"{base_url}/api/v1/actions/{action_id}"
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {access_token}"}
    response = requests.get(url, headers=headers, verify=False)

    print(f"Status Code: {response.status_code}")
    print(f"Response JSON: {response.json()}")
    return response.status_code, response.json()

In [ ]:
# Login
def login(base_url, password):
    url = f"{base_url}/api/v1/login"
    headers = {"Content-Type": "application/json"}
    payload = {"username": "default", "password": password}

    response = requests.post(url, headers=headers, json=payload, verify=False)

    if response.status_code == 200:
        response_json = response.json()
        access_token = response_json['access_token']

    print(f"Status Code: {response.status_code}")
    print(f"Access Token: {access_token}")
    return response.status_code, access_token

In [ ]:
# Status
def get_status(base_url, access_token):
    url = f"{base_url}/api/v1/status"
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {access_token}"}
    response = requests.get(url, headers=headers, verify=False)

    print(f"Status Code: {response.status_code}")
    print(f"Response JSON: {response.json()}")
    return response.json()

In [ ]:
# Deploy Pipeline
def deploy(base_url, access_token, payload):
    url = f"{base_url}/api/v1/pipelines"
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {access_token}"}
    # payload = rdi_deploy_body

    response = requests.post(url, headers=headers, json=payload, verify=False)

    action_id = "NO ACTION"

    if response.status_code == 200:
        response_json = response.json()
        action_id = response_json['action_id']

    print(f"Status Code: {response.status_code}")
    print(f"Response: {response.json()}")
    return action_id

In [ ]:
# Reset Pipeline
def reset(base_url, access_token):
    url = f"{base_url}/api/v1/pipelines/reset"
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {access_token}"}

    response = requests.post(url, headers=headers, verify=False)

    action_id = "NO ACTION"

    if response.status_code == 200:
        response_json = response.json()
        action_id = response_json['action_id']

    print(f"Status Code: {response.status_code}")
    print(f"Response: {response.json()}")
    return action_id

In [ ]:
# GET pipelines
def get_pipeline(base_url, access_token):
    url = f"{base_url}/api/v1/pipelines"
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {access_token}"}
    response = requests.get(url, headers=headers, verify=False)

    print(f"Status Code: {response.status_code}")
    print(f"Response JSON: {response.json()}")
    return response

In [ ]:
# UNDEPLOY Pipeline
def undeploy(base_url, access_token):
    url = f"{base_url}/api/v1/pipelines/undeploy"
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {access_token}"}

    response = requests.post(url, headers=headers, verify=False)

    action_id = "NO ACTION"

    if response.status_code == 200:
        response_json = response.json()
        action_id = response_json['action_id']

    print(f"Status Code: {response.status_code}")
    print(f"Response: {response.json()}")
    return action_id

### Pipeline Definition

In [ ]:
# Pipeline
rdi_deploy_body = {
    "sources": {
        "psql": {
            "type" : "cdc",
            "logging" : {"level": "info"},
            "connection" : {
                "type" : "postgresql",
                "host": SOURCE_DB_IP,
                "port": 5432,
                "user": "redisuser",
                "password": SOURCE_DB_PASSWORD,
                "database": "chinook"
            },
            "tables": {
                "public.artist": {},
                "public.album": {},
                "public.genre": {},
                "public.media_type": {},
                "public.playlist_track": {},
                "public.track": {}
            }
        }
    },
    "targets": {
        "target": {
            "type": "redis",
            "host": TARGET_DB_URL,
            "port": int(TARGET_DB_PORT),
            "password": TARGET_DB_PASSWORD
        }
    },
    "processors": { "on_failed_retry_interval": 5 },    
    "jobs": [
        {
            "name": "artist",
            "source": {"table": "artist"},
            "transform": [{
                "uses": "map",
                "with": {
                    "language": "jmespath",
                    "expression": {
                        "artist_id": "artist_id",
                        "name" : "name"
                        }
                }
            }],
            "output": [
                {
                    "uses": "redis.write",
                    "with": {
                        "connection": "target",
                        "data_type": "json"
                        }
                }
            ]
        },
        {
            "name": "album",
            "source": {"table": "album"},
            "transform":[
                {
                    "uses": "redis.lookup",
                    "with": {
                        "connection": "target",
                        "cmd": "JSON.GET",
                        "args": ["concat(['artist:artist_id:', artist_id])", '`.name`'],
                        "language": "jmespath",
                        "field": "artist"
                    }
                }
            ],
            "output": [
                {
                    "uses": "redis.write",
                    "with": {
                        "connection": "target",
                        "data_type": "json"
                        }
                }
            ]
        },
        {
            "name": "genre",
            "source": {"table": "genre"},
            "transform": [{
                "uses": "map",
                "with": {
                    "language": "jmespath",
                    "expression": {
                        "genre_id": "genre_id",
                        "name" : "name"
                        }
                }
            }],
            "output": [
                {
                    "uses": "redis.write",
                    "with": {
                        "connection": "target",
                        "data_type": "json"
                        }
                }
            ]
        },        
        {
            "name": "track",
            "source": {"table": "track"},
            "transform":[
                {
                    "uses": "filter",
                    "with": {
                        "language": "sql",
                        "expression": "CASE WHEN genre_id != 7 THEN true ELSE False END"
                    }
                },
                {
                    "uses": "redis.lookup",
                    "with": {
                        "connection": "target",
                        "cmd": "JSON.GET",
                        "args": ["concat(['album:album_id:', album_id])", '`.title`'],
                        "language": "jmespath",
                        "field": "album"
                    }
                },
                {
                    "uses": "redis.lookup",
                    "with": {
                        "connection": "target",
                        "cmd": "JSON.GET",
                        "args": ["concat(['album:album_id:', album_id])", '`.artist`'],
                        "language": "jmespath",
                        "field": "artist"
                    }
                },                
                {
                    "uses": "redis.lookup",
                    "with": {
                        "connection": "target",
                        "cmd": "JSON.GET",
                        "args": ["concat(['genre:genre_id:', genre_id])", '`.name`'],
                        "language": "jmespath",
                        "field": "genre"
                    }
                }
            ],
            "output": [
                {
                "uses": "redis.write",
                "with": {
                    "connection": "target",
                    "data_type": "json",
                    "on_update": "merge"
                    }
                },
                {
                "uses": "redis.write",
                "with": {
                    "connection": "target",
                    "data_type": "stream",
                    "key": {
                        "expression": "`track:events`",
                        "language": "jmespath"
                        }
                    }
                }
            ]
        },
    ]
}

### Deploy Pipeline

In [ ]:
status_code, access_token = login(base_url, password)

In [ ]:
status = get_status(base_url, access_token)

In [ ]:
action_id = deploy(base_url, access_token, rdi_deploy_body)

In [ ]:
status_code, response = get_action(base_url, action_id, access_token)

In [ ]:
# Use reset only if you need to repopulate Redis based on pipeline changes
# action_id = reset(base_url, access_token)

In [ ]:
# status_code, response = get_action(base_url, action_id, access_token)